# VST DenseDet Experiment

Train and evaluate the DenseDet pipeline with the custom `VSTBackbone`.

Update the dataset and save paths below for Colab or Kaggle before running.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

if Path('/kaggle/working/SLIM-DET').exists():
    REPO_DIR = Path('/kaggle/working/SLIM-DET')
    DATASET_ROOT = Path('/root/.cache/kagglehub/datasets/sakettiw11/aircraft-dataset/versions/1/content/Aircraft_dataset')
    SAVE_DIR = Path('/kaggle/working/dense_det_vst')
elif Path('/content/SLIM-DET').exists():
    REPO_DIR = Path('/content/SLIM-DET')
    DATASET_ROOT = Path('/content/dataset/content/content/Aircraft_dataset')
    SAVE_DIR = Path('/content/drive/MyDrive/slim_det_runs/dense_det_vst')
else:
    raise FileNotFoundError('Clone the SLIM-DET repo first.')

print('REPO_DIR   =', REPO_DIR)
print('DATASET_ROOT =', DATASET_ROOT)
print('SAVE_DIR   =', SAVE_DIR)

In [ ]:
requirements = REPO_DIR / 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)

In [ ]:
train_cmd = [
    sys.executable,
    '-u',
    'train_dense.py',
    '--config', 'configs/dense_det_vst.yaml',
    '--data_format', 'detection',
    '--dataset_root', str(DATASET_ROOT),
    '--backbone_name', 'vst',
    '--variant', 'small',
    '--neck_name', 'cafpn',
    '--quality_head',
    '--balanced_sampler',
    '--assigner', 'fcos',
    '--epochs', '300',
    '--batch', '8',
    '--imgsz', '640',
    '--workers', '2',
    '--eval_every_epochs', '5',
    '--save_every_batches', '200',
    '--save_dir', str(SAVE_DIR),
]

print(' '.join(train_cmd))
subprocess.run(train_cmd, cwd=str(REPO_DIR), check=True)

In [ ]:
eval_cmd = [
    sys.executable,
    'evaluate_dense.py',
    '--config', 'configs/dense_det_vst.yaml',
    '--data_format', 'detection',
    '--dataset_root', str(DATASET_ROOT),
    '--checkpoint', str(SAVE_DIR / 'dense_det_last.pt'),
    '--batch', '8',
    '--imgsz', '640',
    '--workers', '2',
]

print(' '.join(eval_cmd))
subprocess.run(eval_cmd, cwd=str(REPO_DIR), check=True)